<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset
from PIL import Image
import numpy as np
import random

c:\Users\Balint\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
model.eval();

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
from torchvision.datasets import OxfordIIITPet
import os

root = "./oxford_pets"
os.makedirs(root, exist_ok=True)

trainval = OxfordIIITPet(root=root, split="trainval", download=True)
test = OxfordIIITPet(root=root, split="test", download=True)

CLASSES = trainval.classes
print(f"Classes ({len(CLASSES)}): {CLASSES}")

class_to_protos = {c: [] for c in CLASSES}
for img, lbl in trainval:
    class_to_protos[CLASSES[lbl]].append(img.convert("RGB"))

eval_set = [(img.convert("RGB"), CLASSES[lbl]) for img, lbl in test]
random.shuffle(eval_set)

eval_images = [img for img, _ in eval_set]
eval_labels = [c for _, c in eval_set]

Classes (37): ['Abyssinian', 'American Bulldog', 'American Pit Bull Terrier', 'Basset Hound', 'Beagle', 'Bengal', 'Birman', 'Bombay', 'Boxer', 'British Shorthair', 'Chihuahua', 'Egyptian Mau', 'English Cocker Spaniel', 'English Setter', 'German Shorthaired', 'Great Pyrenees', 'Havanese', 'Japanese Chin', 'Keeshond', 'Leonberger', 'Maine Coon', 'Miniature Pinscher', 'Newfoundland', 'Persian', 'Pomeranian', 'Pug', 'Ragdoll', 'Russian Blue', 'Saint Bernard', 'Samoyed', 'Scottish Terrier', 'Shiba Inu', 'Siamese', 'Sphynx', 'Staffordshire Bull Terrier', 'Wheaten Terrier', 'Yorkshire Terrier']


In [5]:
from tqdm import tqdm
@torch.no_grad()
def encode_images(pil_images, batch_size=32):
    feats = []
    for i in tqdm(range(0, len(pil_images), batch_size)):
        batch = pil_images[i:min(i+batch_size, len(pil_images))]
        inputs = processor(images=batch, return_tensors="pt").to(device)
        encoded = model.get_image_features(**inputs).cpu()
        feats.append(encoded)
    return F.normalize(torch.cat(feats), dim=1)


eval_images = [img for img, _ in eval_set]
eval_labels = [c for _, c in eval_set]

eval_features = encode_images(eval_images)
print(f"Eval features: {eval_features.shape}")

100%|██████████| 115/115 [00:16<00:00,  7.18it/s]

Eval features: torch.Size([3669, 512])


In [6]:
@torch.no_grad()
def encode_texts(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    return F.normalize(model.get_text_features(**inputs), dim=1)

def accuracy(pred_classes, true_classes):
    return (pred_classes == true_classes).float().sum() / len(true_classes)

texts = [f"Photo of a {class_name}" for class_name in CLASSES]
text_embeds = encode_texts(texts).cpu()
class_to_idx = {CLASSES[i]:i for i in range(len(CLASSES))}
true_labels = torch.tensor([class_to_idx[c] for c in eval_labels])
preds = (text_embeds @ eval_features.T).argmax(0)
pred_labels = [CLASSES[i] for i in preds]
accuracy(true_labels, preds)

tensor(0.8523)

In [7]:
from collections import Counter

errors = [(t, p) for t, p in zip(eval_labels, pred_labels) if t != p]
print(f"Total errors: {len(errors)} / {len(eval_labels)}")
print("\nMost common confusions (true -> pred):")
for (t, p), n in Counter(errors).most_common(10):
    print(f"  {t:30s} -> {p:30s}  ({n})")

Total errors: 542 / 3669

Most common confusions (true -> pred):
  Ragdoll                        -> Birman                          (64)
  Bombay                         -> Russian Blue                    (50)
  Boxer                          -> American Bulldog                (40)
  Persian                        -> Birman                          (30)
  American Pit Bull Terrier      -> American Bulldog                (21)
  British Shorthair              -> Russian Blue                    (19)
  Bombay                         -> Siamese                         (18)
  English Cocker Spaniel         -> English Setter                  (17)
  Staffordshire Bull Terrier     -> American Bulldog                (14)
  Beagle                         -> Basset Hound                    (12)


In [8]:
@torch.no_grad()
def encode_train_images():
    X_train, y_train = [], []
    for x, y in tqdm(trainval):
        x = x.convert("RGB")
        y_train.append(y)
        inputs = processor(images=x, return_tensors="pt").to(device)
        encoded = model.get_image_features(**inputs).cpu()
        X_train.append(encoded)
    return X_train, y_train

In [9]:
X_train, y_train = encode_train_images()

100%|██████████| 3680/3680 [00:34<00:00, 105.41it/s]


In [ ]:
X_train, y_train = np.stack(X_train),  np.stack(y_train)
X_val = eval_features.numpy()
y_val = true_labels.numpy()
mean = X_train.mean(axis=1)
std = X_train.std(axis=1)
X_train = (X_train - mean) / std

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

lg = LogisticRegression().fit(X_train, y_train)
train_preds, val_preds = lg.predict(X_train), lg.predict(y_train)
accuracy_score(y_train, train_preds), accuracy_score(y_val, val_preds), f1_score(y_train, train_preds, average='macro'), f1_score(y_val, val_preds, average='macro')

((3680, 512), (3680,))